# Predicting Sales

Preparing the data for machine learning, creating the relevant features, splitting the data in time order into training, validation and test,
and building a baseline model plus Random Forest and Gradient Boosting models. Comparing the performance with MAE, RMSE and WAPE,
visualising the predictions, analysing the errors and explaining the limits of the models.


In [1]:
from pathlib import Path
from google.colab import files

REQUIRED_FILES = ['model_dataset.csv']
missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if missing:
    print('Select these CSV files:', ', '.join(missing))
    uploaded = files.upload()
else:
    print('All required CSV files are already uploaded.')

still_missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if still_missing:
    raise FileNotFoundError('Missing CSV files: ' + ', '.join(still_missing))
print('Ready! All required CSV files are present.')

All required CSV files are already uploaded.
Ready! All required CSV files are present.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
TABLES = PROJECT_ROOT / 'reports' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

def read_data(folder, filename):
    path = PROJECT_ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing {filename}. Run the upload cell at the top of this notebook.')
    df = pd.read_csv(path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    return df

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=150, bbox_inches='tight')
    plt.close()

def save_table(df, filename):
    df.to_csv(TABLES / filename, index=False)

def check_close(actual, expected, label, tolerance=0.06):
    if not np.isclose(actual, expected, atol=tolerance):
        raise ValueError(f'{label}: {actual:.2f} != {expected:.2f}')

def quality_report(datasets):
    for name, df in datasets.items():
        print(f'{name}: {len(df)} rows; missing={df.isna().sum().sum()}; duplicates={df.duplicated().sum()}')

pd.set_option('display.float_format', '{:,.2f}'.format)
print('Working folder:', PROJECT_ROOT)


from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURES = ['hour', 'weekday', 'is_weekend', 'day_of_year', 'month_number']
TARGET = 'quantity'

def load_model_data():
    df = read_data('processed', 'model_dataset.csv')
    return df.sort_values(['date', 'hour']).reset_index(drop=True)

def add_features(df):
    df = df.copy()
    df['weekday'] = df['date'].dt.weekday
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)
    df['day_of_year'] = df['date'].dt.dayofyear
    df['month_number'] = df['date'].dt.month
    return df

def split_by_date(df):
    dates = np.sort(df['date'].unique())
    first = pd.Timestamp(dates[int(len(dates) * .60)])
    second = pd.Timestamp(dates[int(len(dates) * .80)])
    return (df['date'] < first,
            (df['date'] >= first) & (df['date'] < second),
            df['date'] >= second, first, second)

def baseline_predict(train_df, predict_df):
    lookup = train_df.groupby(['weekday', 'hour'])[TARGET].mean()
    fallback = train_df[TARGET].mean()
    return np.array([lookup.get((int(w), int(h)), fallback)
                     for w, h in zip(predict_df['weekday'], predict_df['hour'])])

def scores(actual, predicted):
    a, p = np.asarray(actual), np.asarray(predicted)
    return {'MAE': mean_absolute_error(a, p),
            'RMSE': np.sqrt(mean_squared_error(a, p)),
            'WAPE_pct': 100 * np.abs(a-p).sum() / a.sum() if a.sum() else np.nan}

def get_models():
    return {'Random forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=3,
                                                   random_state=42, n_jobs=-1),
            'Gradient boosting': HistGradientBoostingRegressor(max_iter=120,
                                                               max_leaf_nodes=15, random_state=42)}


Working folder: /content


## 1. Load and validate

In [3]:
model = load_model_data()
print("Rows:", len(model), "| Columns:", len(model.columns))

assert model[["date", "hour"]].duplicated().sum() == 0, "Duplicate date/hour rows"
assert model["quantity"].ge(0).all(), "Negative demand"
model.head()

Rows: 1170 | Columns: 9


,date,hour,weekday,quantity,revenue,is_weekend,month,annual_inflation_pct,avg_net_wage_ron
0,2026-01-01,7,3,19,136,0,2026-01,9.60,5518
1,2026-01-01,8,3,27,177,0,2026-01,9.60,5518
2,2026-01-01,9,3,33,224,0,2026-01,9.60,5518
3,2026-01-01,10,3,25,173,0,2026-01,9.60,5518
4,2026-01-01,11,3,23,163,0,2026-01,9.60,5518


## 2. Features

In [4]:
model = add_features(model)
print("Features:", FEATURES)
model[FEATURES + [TARGET]].describe()

Features: ['hour', 'weekday', 'is_weekend', 'day_of_year', 'month_number']


,hour,weekday,is_weekend,day_of_year,month_number,quantity
count,"1,170.00","1,170.00","1,170.00","1,170.00","1,170.00","1,170.00"
mean,13.00,3.01,0.29,45.50,2.00,18.73
std,3.74,2.01,0.45,25.99,0.83,8.15
min,7.00,0.00,0.00,1.00,1.00,1.00
25%,10.00,1.00,0.00,23.00,1.00,13.00
50%,13.00,3.00,0.00,45.50,2.00,18.00
75%,16.00,5.00,1.00,68.00,3.00,23.00
max,19.00,6.00,1.00,90.00,3.00,56.00


In [5]:
# seasonality: what patterns can the models learn?
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
model.groupby("hour")["quantity"].mean().plot.bar(ax=axes[0], color="peru", title="by hour")
model.groupby("weekday")["quantity"].mean().plot.bar(ax=axes[1], color="sienna", title="by weekday (0 = Monday)")
model.groupby("month_number")["quantity"].mean().plot.bar(ax=axes[2], color="tan", title="by month")
save_plot("forecast_seasonality.png")

## 3. Chronological split

In [6]:
train_mask, validation_mask, test_mask, train_end, validation_end = split_by_date(model)

print("Training rows:", train_mask.sum())
print("Validation rows:", validation_mask.sum(), "| starts", train_end.date())
print("Test rows:", test_mask.sum(), "| starts", validation_end.date())

Training rows: 702
Validation rows: 234 | starts 2026-02-24
Test rows: 234 | starts 2026-03-14


## 4. Baseline and models, compared on the validation set

In [7]:
train_df = model[train_mask]
validation_df = model[validation_mask]

validation_results = []

# baseline: average of the same weekday and hour in the training data
baseline_pred = baseline_predict(train_df, validation_df)
validation_results.append({"model": "Weekday/hour baseline", **scores(validation_df[TARGET], baseline_pred)})

models = get_models()
for name, estimator in models.items():
    estimator.fit(train_df[FEATURES], train_df[TARGET])
    pred = np.maximum(0, estimator.predict(validation_df[FEATURES]))   # demand cannot be negative
    validation_results.append({"model": name, **scores(validation_df[TARGET], pred)})

validation_table = pd.DataFrame(validation_results).sort_values("MAE").reset_index(drop=True)
validation_table

,model,MAE,RMSE,WAPE_pct
0,Weekday/hour baseline,3.82,4.90,19.61
1,Gradient boosting,3.83,4.98,19.68
2,Random forest,3.93,5.13,20.16


In [8]:
selected_model = validation_table.iloc[0]["model"]
print("Selected model (lowest validation MAE):", selected_model)

validation_table.plot.bar(x="model", y=["MAE", "RMSE"], figsize=(9, 5), rot=0,
                          title="Validation error (lower is better)", color=["peru", "saddlebrown"])
plt.xlabel("")
plt.ylabel("items per hour")
save_plot("forecast_validation_scores.png")

Selected model (lowest validation MAE): Weekday/hour baseline


## 5. Final test

In [9]:
# now I train the selected model on train + validation and test it once
fit_mask = train_mask | validation_mask
fit_df = model[fit_mask]
test_df = model[test_mask]

if selected_model == "Weekday/hour baseline":
    test_pred = baseline_predict(fit_df, test_df)
else:
    final_model = models[selected_model]
    final_model.fit(fit_df[FEATURES], fit_df[TARGET])
    test_pred = np.maximum(0, final_model.predict(test_df[FEATURES]))

test_scores = scores(test_df[TARGET], test_pred)
for metric, value in test_scores.items():
    print(f"{metric}: {value:.3f}")

MAE: 3.501
RMSE: 4.517
WAPE_pct: 18.191


## 6. Where is the model wrong?

In [10]:
results = test_df[["date", "hour", "quantity"]].copy()
results["predicted_items"] = test_pred
results["absolute_error"] = (results["quantity"] - results["predicted_items"]).abs()

results.groupby("hour")["absolute_error"].mean().plot.bar(figsize=(9, 4), color="tan", rot=0,
                                                         title="Average error by hour (items)")
save_plot("forecast_error_by_hour.png")

In [11]:
first_week = results[results["date"] < results["date"].min() + pd.Timedelta(days=7)]

plt.figure(figsize=(12, 5))
plt.plot(np.arange(len(first_week)), first_week["quantity"].to_numpy(), label="Observed demand", color="black")
plt.plot(np.arange(len(first_week)), first_week["predicted_items"].to_numpy(), label="Predicted demand", color="peru")
plt.title("Hourly Demand: Actual vs Predicted (first test week)")
plt.xlabel("Opening hours in sequence")
plt.ylabel("Items sold")
plt.legend()
save_plot("forecast_test_week.png")

In [12]:
# daily totals are easier to judge than single hours
daily_results = results.groupby("date")[["quantity", "predicted_items"]].sum()
daily_error_pct = ((daily_results["quantity"] - daily_results["predicted_items"]).abs() / daily_results["quantity"] * 100).mean()
print(f"Average error on a whole day: {daily_error_pct:.1f}%")

Average error on a whole day: 7.0%


## 7. Export the tables

In [13]:
save_table(validation_table, "forecast_validation_scores.csv")
save_table(results, "forecast_test_predictions.csv")

## What I found
Demand can be predicted well enough to plan with: about 3.5 items per hour of error, and about 7% on a whole day.
The surprise is that the simple baseline (the average of the same weekday and hour) was as good as the two machine learning models.
In this data the hour and the weekday explain most of the demand, and the rest is random noise.